In [1]:
from pathlib import Path

import xarray as xr
import panel as pn
from holoviews import opts

import echoshader

In [2]:
path_MVBS = Path("/Users/wujung/Downloads/prefect_sh2506_MVBS")
MVBS_files = sorted(list(path_MVBS.glob("MVBS_*.zarr")))

In [3]:
MVBS_files[-2]

PosixPath('/Users/wujung/Downloads/prefect_sh2506_MVBS/MVBS_20250618T150000.zarr')

In [4]:
ds_MVBS = xr.open_zarr(MVBS_files[-2])

In [5]:
ds_MVBS

<xarray.Dataset> Size: 7MB
Dimensions:            (channel: 5, ping_time: 240, depth: 757)
Coordinates:
  * channel            (channel) <U25 500B 'WBT 400140-15 ES120-7C_ES' ... 'W...
  * depth              (depth) float64 6kB 0.0 1.0 2.0 3.0 ... 754.0 755.0 756.0
  * ping_time          (ping_time) datetime64[ns] 2kB 2025-06-18T15:00:00 ......
Data variables:
    Sv                 (channel, ping_time, depth) float64 7MB dask.array<chunksize=(5, 240, 757), meta=np.ndarray>
    frequency_nominal  (channel) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    latitude           (ping_time) float64 2kB dask.array<chunksize=(240,), meta=np.ndarray>
    longitude          (ping_time) float64 2kB dask.array<chunksize=(240,), meta=np.ndarray>
Attributes:
    processing_function:          commongrid.compute_MVBS
    processing_level:             Level 3A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...
    processing_software_name:     echopype
    processing_software_version:  0.10.1
    processing_time:              2025-06-18T15:43:03Z

In [6]:
ds_MVBS["echo_range"] = ds_MVBS["depth"]
ds_MVBS = ds_MVBS.swap_dims({"depth": "echo_range"})
ds_MVBS["Sv"] = ds_MVBS["Sv"].assign_attrs(
    actual_range=(float(ds_MVBS["Sv"].min().compute()),
                  float(ds_MVBS["Sv"].max().compute()))
)

In [7]:
ds_MVBS

<xarray.Dataset> Size: 7MB
Dimensions:            (channel: 5, ping_time: 240, echo_range: 757)
Coordinates:
  * channel            (channel) <U25 500B 'WBT 400140-15 ES120-7C_ES' ... 'W...
    depth              (echo_range) float64 6kB 0.0 1.0 2.0 ... 755.0 756.0
  * ping_time          (ping_time) datetime64[ns] 2kB 2025-06-18T15:00:00 ......
  * echo_range         (echo_range) float64 6kB 0.0 1.0 2.0 ... 755.0 756.0
Data variables:
    Sv                 (channel, ping_time, echo_range) float64 7MB dask.array<chunksize=(5, 240, 757), meta=np.ndarray>
    frequency_nominal  (channel) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    latitude           (ping_time) float64 2kB dask.array<chunksize=(240,), meta=np.ndarray>
    longitude          (ping_time) float64 2kB dask.array<chunksize=(240,), meta=np.ndarray>
Attributes:
    processing_function:          commongrid.compute_MVBS
    processing_level:             Level 3A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...
    processing_software_name:     echopype
    processing_software_version:  0.10.1
    processing_time:              2025-06-18T15:43:03Z

In [8]:
ds_MVBS["channel"]

<xarray.DataArray 'channel' (channel: 5)> Size: 500B
array(['WBT 400140-15 ES120-7C_ES', 'WBT 400141-15 ES18_ES',
       'WBT 400142-15 ES70-7C_ES', 'WBT 400143-15 ES38B_ES',
       'WBT 400145-15 ES200-7C_ES'], dtype='<U25')
Coordinates:
  * channel  (channel) <U25 500B 'WBT 400140-15 ES120-7C_ES' ... 'WBT 400145-...
Attributes:
    long_name:  Vendor channel ID

In [9]:
egram = ds_MVBS.eshader.echogram(
    channel=[
        "WBT 400141-15 ES18_ES",
        "WBT 400143-15 ES38B_ES",
        "WBT 400142-15 ES70-7C_ES",
        "WBT 400140-15 ES120-7C_ES",
        "WBT 400145-15 ES200-7C_ES",
    ],
    vmin=-70,
    vmax=-36,
    cmap = "viridis",
    opts = opts.Image(
        width=800, height=400
    )
)

In [10]:
tricolor = ds_MVBS.eshader.echogram(
    channel=[
        "WBT 400140-15 ES120-7C_ES",
        "WBT 400143-15 ES38B_ES",
        "WBT 400141-15 ES18_ES",
    ],
    vmin=-70,
    vmax=-36,
    rgb_composite=True,
    opts=opts.RGB(width=800)
)

In [11]:
test_server = pn.serve(
    {
        "test": pn.Column(egram()),
        "tricolor": pn.Column(tricolor()),
    },
    port=1802,
    # websocket_origin="*",
    admin=True,
    # show=False
)


Launching server at http://localhost:1802


In [14]:
test_server.stop()